# 🗑️ Garbage Classification with Naive Bayes
**Dataset:** Garbage Classification (Kaggle — mostafaabla/garbage-classification)
**Goal:** Classify waste items as *Recyclable* or *Non-Recyclable* using Naive Bayes

---
### Setup
```bash
pip install -r requirements.txt
python download_dataset.py      # OR
python generate_sample_dataset.py
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
%matplotlib inline
sns.set_theme(style='whitegrid')
print('Libraries loaded ✅')

## 1 — Load Dataset

In [ ]:
df = pd.read_csv('data/garbage_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Category distribution
df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Samples per Category'); axes[0].set_xlabel(''); axes[0].tick_params(rotation=30)

# Binary label
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=['#2ecc71','#e74c3c'], startangle=90)
axes[1].set_title('Recyclable vs Non-Recyclable'); axes[1].set_ylabel('')
plt.tight_layout()

## 2 — Feature Exploration

In [ ]:
feature_cols = [c for c in df.columns if c not in ('filename','category','label')]
print('Feature columns:', feature_cols)
df[feature_cols].describe()

In [ ]:
# Feature distributions per category
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = sns.color_palette('tab10', 6)
for ax, feat in zip(axes.flat, feature_cols):
    for color, cat in zip(colors, df['category'].unique()):
        df[df['category']==cat][feat].hist(ax=ax, alpha=0.5, bins=20, color=color, label=cat)
    ax.set_title(feat); ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Category', fontsize=13, y=1.01)
plt.tight_layout()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8,5))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()

## 3 — Bayes' Theorem — How It Works

In [ ]:
print('Bayes\' Theorem:')
print('P(Class | Features) = P(Features | Class) × P(Class) / P(Features)')
print()

# Manual prior calculation
label_counts = df['label'].value_counts()
n_total = len(df)

for label, count in label_counts.items():
    print(f'P({label:20s}) = {count}/{n_total} = {count/n_total:.4f}')

## 4 — Train Naive Bayes Models

In [ ]:
X = df[feature_cols].values
y = (df['label'] == 'Recyclable').astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

In [ ]:
nb = GaussianNB()
nb.fit(X_train_s, y_train)
y_pred = nb.predict(X_test_s)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Non-Recyclable','Recyclable']))

## 5 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Recyclable','Recyclable'],
            yticklabels=['Non-Recyclable','Recyclable'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()

## 6 — Cross-Validation

In [ ]:
cv_scores = cross_val_score(GaussianNB(), X_train_s, y_train, cv=5, scoring='accuracy')
print(f'CV Scores  : {cv_scores}')
print(f'Mean ± Std : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

plt.figure(figsize=(6,3))
plt.bar(range(1,6), cv_scores, color='steelblue', edgecolor='white')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean={cv_scores.mean():.3f}')
plt.xlabel('Fold'); plt.ylabel('Accuracy'); plt.title('5-Fold Cross-Validation')
plt.legend(); plt.tight_layout()

## 7 — Predict a New Item

In [ ]:
# Adjust values to describe your item
new_item = {
    'material_hardness':  0.55,   # 0=soft, 1=hard
    'transparency':       0.45,   # 0=opaque, 1=transparent
    'surface_smoothness': 0.80,   # 0=rough, 1=smooth
    'estimated_weight':   0.25,   # 0=light, 1=heavy
    'organic_content':    0.02,   # 0=none, 1=high
    'color_uniformity':   0.65,   # 0=varied, 1=uniform
}

x_new = np.array(list(new_item.values())).reshape(1,-1)
x_new_s = scaler.transform(x_new)
pred = nb.predict(x_new_s)[0]
prob = nb.predict_proba(x_new_s)[0]

label = 'Recyclable' if pred == 1 else 'Non-Recyclable'
print(f'Prediction  : {label}')
print(f'Confidence  : {max(prob)*100:.1f}%')